In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# Replace with your actual GitHub username
GITHUB_USER="jyotishmann"

git clone https://github.com/$GITHUB_USER/docusage.git /content/docusage
cd /content/docusage
pip install -r requirements.txt -q

In [ ]:
import os

# All data lives on Drive so it survives restarts
DRIVE_DATA = "/content/drive/MyDrive/docusage_data"
os.makedirs(DRIVE_DATA, exist_ok=True)

# These env vars are picked up by config/settings.py
os.environ["DATA_DIR"]      = DRIVE_DATA
os.environ["DEVICE"]        = "cuda"        # T4 GPU
os.environ["GRADIO_SHARE"]  = "true"        # public gradio.live URL
os.environ["LOG_LEVEL"]     = "INFO"

print("Data directory:", DRIVE_DATA)
print("GPU available:", __import__('torch').cuda.is_available())

In [ ]:
from pathlib import Path
import subprocess, sys

index_dir = Path(DRIVE_DATA) / "indices"
registry  = Path(DRIVE_DATA) / "corpus_registry.json"

if index_dir.exists() and any(index_dir.glob("*.pkl")):
    print("✅ Indices found on Drive — skipping build.")
    print(f"   Files: {list(index_dir.iterdir())}")
else:
    print("⏳ First-time setup: downloading corpus and building indices.")
    print("   This takes ~15-25 minutes. It will NOT run again next session.\n")

    # Download Phase 1 corpus
    result = subprocess.run(
        [sys.executable, "scripts/download_corpus.py", "--phase", "1"],
        cwd="/content/docusage", capture_output=False
    )

    # Build BM25 + FAISS indices
    result = subprocess.run(
        [sys.executable, "scripts/build_index.py"],
        cwd="/content/docusage", capture_output=False
    )
    print("\n✅ Index build complete. Saved to Drive.")

In [ ]:
import sys, os
sys.path.insert(0, "/content/docusage")
os.chdir("/content/docusage")

# Import the app (this loads indices eagerly, models lazily)
from app import demo

# share=True is read from GRADIO_SHARE env var via settings
demo.launch(
    share=True,
    show_api=False,
    quiet=False,
)

# Prints two lines:
#   Local URL:  http://127.0.0.1:7860
#   Public URL: https://xxxx.gradio.live   <-- share this one